In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


In [2]:
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv",sep=";")

In [3]:
df

,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.60
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.50
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.60
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.60
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20164,31/12/2024,31,12,1,4,1,Tuesday,TN0008000812,"BTA 7,5% 13/12/2028",215.0,0.200,50.0,9.00
20165,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,5243.0,0.556,15.0,7.49
20166,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,28288.0,3.000,31.0,8.99
20167,31/12/2024,31,12,1,4,1,Tuesday,TNX0K9990B08,EMP NAT 2024 T2 CB TF,48701.0,5.165,31.0,8.99


In [4]:
target = 'Montant'

# Sélection des colonnes numériques comme features (tu peux ajuster cette liste)
features = df.select_dtypes(include=[np.number]).columns.tolist()

# Supprimer la colonne target des features
features.remove(target)

X = df[features]
y = df[target]

print(f"Features utilisées ({len(features)}): {features}")
print(f"Nombre d'exemples : {len(df)}")


Features utilisées (8): ['Jour', 'Mois', 'NumeroSemaine', 'Trimestre', 'JourSemaineNum', 'Nombre de Titres', 'Echéance', 'Taux']
Nombre d'exemples : 20169


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size: {X_test.shape[0]} samples")


Train size: 16135 samples
Test size: 4034 samples


In [6]:
def eval_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return rmse


In [7]:
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
}


In [8]:
results = {}

for name, model in models.items():
    print(f"\n=== Modèle : {name} ===")
    rmse_raw = eval_model(model, X_train, X_test, y_train, y_test)
    print(f"RMSE brut: {rmse_raw:.4f}")
    results[name] = {'rmse_brut': rmse_raw}



=== Modèle : LinearRegression ===
RMSE brut: 7.0118

=== Modèle : Ridge ===
RMSE brut: 7.0118

=== Modèle : Lasso ===
RMSE brut: 7.0207

=== Modèle : DecisionTree ===
RMSE brut: 4.6152

=== Modèle : RandomForest ===
RMSE brut: 3.6171

=== Modèle : GradientBoosting ===
RMSE brut: 4.6012


In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for name, model in models.items():
    print(f"\n=== Modèle : {name} ===")
    rmse_scaled = eval_model(model, X_train_scaled, X_test_scaled, y_train, y_test)
    print(f"RMSE après normalisation: {rmse_scaled:.4f}")
    results[name]['rmse_scaled'] = rmse_scaled



=== Modèle : LinearRegression ===
RMSE après normalisation: 7.0118

=== Modèle : Ridge ===
RMSE après normalisation: 7.0118

=== Modèle : Lasso ===
RMSE après normalisation: 7.0804

=== Modèle : DecisionTree ===
RMSE après normalisation: 4.6680

=== Modèle : RandomForest ===
RMSE après normalisation: 3.6185

=== Modèle : GradientBoosting ===
RMSE après normalisation: 4.6016


In [14]:
k = min(3, X_train.shape[1])  # Choix de k features à garder

selector = SelectKBest(score_func=f_regression, k=k)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel = selector.transform(X_test_scaled)

print(f"Nombre de features sélectionnées : {k}")

for name, model in models.items():
    print(f"\n=== Modèle : {name} ===")
    rmse_sel = eval_model(model, X_train_sel, X_test_sel, y_train, y_test)
    print(f"RMSE après sélection de features: {rmse_sel:.4f}")
    results[name]['rmse_selection'] = rmse_sel


Nombre de features sélectionnées : 3

=== Modèle : LinearRegression ===
RMSE après sélection de features: 7.0180

=== Modèle : Ridge ===
RMSE après sélection de features: 7.0180

=== Modèle : Lasso ===
RMSE après sélection de features: 7.0804

=== Modèle : DecisionTree ===
RMSE après sélection de features: 5.1242

=== Modèle : RandomForest ===
RMSE après sélection de features: 3.9680

=== Modèle : GradientBoosting ===
RMSE après sélection de features: 4.6408


In [15]:
# Hyperparamètres à tester par modèle
param_grids = {
    'Ridge': {'alpha': [0.01, 0.1, 1, 10, 100]},
    'RandomForest': {'n_estimators': [50, 100], 'max_depth': [None, 10, 20]},
    'GradientBoosting': {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1], 'max_depth': [3,5]},
}

for name, model in models.items():
    print(f"\n=== Modèle : {name} ===")

    if name in param_grids:
        grid = GridSearchCV(model, param_grids[name], cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
        grid.fit(X_train_sel, y_train)
        best_model = grid.best_estimator_
        preds_ft = best_model.predict(X_test_sel)
        rmse_ft = np.sqrt(mean_squared_error(y_test, preds_ft))
        print(f"Meilleurs params: {grid.best_params_}")
        print(f"RMSE après fine tuning: {rmse_ft:.4f}")
        results[name]['rmse_finetuning'] = rmse_ft
        results[name]['best_params'] = grid.best_params_
    else:
        print("Fine tuning non applicable")
        results[name]['rmse_finetuning'] = None
        results[name]['best_params'] = None



=== Modèle : LinearRegression ===
Fine tuning non applicable

=== Modèle : Ridge ===
Meilleurs params: {'alpha': 10}
RMSE après fine tuning: 7.0180

=== Modèle : Lasso ===
Fine tuning non applicable

=== Modèle : DecisionTree ===
Fine tuning non applicable

=== Modèle : RandomForest ===
Meilleurs params: {'max_depth': 20, 'n_estimators': 100}
RMSE après fine tuning: 3.9584

=== Modèle : GradientBoosting ===
Meilleurs params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
RMSE après fine tuning: 4.0843


In [16]:
import pprint
print("\n=== Résumé complet des RMSE par modèle ===")
pprint.pprint(results)



=== Résumé complet des RMSE par modèle ===
{'DecisionTree': {'best_params': None,
                  'rmse_brut': 4.615179683227573,
                  'rmse_finetuning': None,
                  'rmse_scaled': 4.668045853091294,
                  'rmse_selection': 5.124166995402363},
 'GradientBoosting': {'best_params': {'learning_rate': 0.1,
                                      'max_depth': 5,
                                      'n_estimators': 100},
                      'rmse_brut': 4.601168612666219,
                      'rmse_finetuning': 4.084340632713811,
                      'rmse_scaled': 4.601630229306337,
                      'rmse_selection': 4.640782469065166},
 'Lasso': {'best_params': None,
           'rmse_brut': 7.020682544276065,
           'rmse_finetuning': None,
           'rmse_scaled': 7.080448595879989,
           'rmse_selection': 7.080448595879989},
 'LinearRegression': {'best_params': None,
                      'rmse_brut': 7.011799630094477,
          